In [1]:
# === Run this cell FIRST (works even if the notebook is inside /Model) ===
import os, sys, pathlib

# Add the project root (the folder that contains `project_package/`) to Python path
def add_project_root(marker_folder="project_package", max_up=5):
    cur = pathlib.Path(".").resolve()
    for _ in range(max_up + 1):
        if (cur / marker_folder).is_dir():
            if str(cur) not in sys.path:
                sys.path.insert(0, str(cur))
            return cur
        cur = cur.parent
    raise RuntimeError(f"Could not find '{marker_folder}' up to {max_up} levels above.")

PROJECT_ROOT = add_project_root()
print("PROJECT_ROOT =", PROJECT_ROOT)

# Resolve CSV location (here we assume it's at the project root; move to /datasets if you prefer)
CSV_NAME = "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
csv_path = PROJECT_ROOT / CSV_NAME
print("CSV exists:", csv_path.exists(), "->", csv_path)

# Import unsupervised runners from your package
from project_package.modeling import (
    run_kmeans_from_csv,
    run_isolation_forest_from_csv,
    run_pca_embeddings_from_csv,
)

# ID columns to carry through outputs (useful for later joins/plots)
IDS = ["Booking ID", "Customer ID"]

# --- 1) KMeans clustering ---
# Tries multiple k, picks the best by silhouette; exports labels and PCA(2D) CSVs.
km = run_kmeans_from_csv(
    csv_path=str(csv_path),
    id_cols=IDS,
    artifacts_dir="unsupervised",   # will be saved under <project_root>/artifacts/unsupervised/
    k_list=[3, 5, 8, 10],
    random_state=42,
)
print("Best k:", km.best_k)
print("Quality per k:", km.report)
print("Labels CSV:", km.labels_csv_path)
print("PCA(2D) CSV:", km.pca2_csv_path)

# --- 2) Isolation Forest (anomaly detection) ---
# Flags unusual rows; contamination ≈ expected outlier fraction.
iso = run_isolation_forest_from_csv(
    csv_path=str(csv_path),
    id_cols=IDS,
    artifacts_dir="unsupervised",
    contamination=0.02,
    random_state=42,
)
print("Anomaly scores CSV:", iso.scores_csv_path)

# --- 3) PCA embeddings (2D) ---
# Produces 2D coordinates (pc1, pc2) for easy scatter plotting.
emb = run_pca_embeddings_from_csv(
    csv_path=str(csv_path),
    id_cols=IDS,
    artifacts_dir="unsupervised",
    n_components=2,
    random_state=42,
)
print("PCA embeddings CSV:", emb.embed_csv_path)


PROJECT_ROOT = C:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis
CSV exists: True -> C:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\ncr_ride_bookings_with_weather_filled_scaled_short.csv
Best k: 5
Quality per k: {3: {'silhouette': 0.3554688933675589, 'dbi': 0.9054461774345682, 'ch': 104490.28537339362}, 5: {'silhouette': 0.35646069636679517, 'dbi': 0.9147072898517564, 'ch': 120779.83136480674}, 8: {'silhouette': 0.3160060099867677, 'dbi': 0.9037372796762839, 'ch': 118250.41199827858}, 10: {'silhouette': 0.31093862008006423, 'dbi': 0.9470420335286249, 'ch': 113987.0449571004}}
Labels CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5_labels.csv
PCA(2D) CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\kmeans_k5_pca2.csv
Anomaly scores CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\isoforest_scores.csv
PCA embeddings CSV: c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\unsupervised\pca_2d.csv
